# Fix Critical Data Issues

**Author: Rohan Raval**

**Objectives:**
1. Fix ACS notebook KeyError
2. Geocode missing library and YMCA coordinates
3. Enhance youth-serving identification

In [6]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import time

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 150)

candidates = [Path("."), Path(".."), Path("../..")]
PROJECT_ROOT = None
for cand in candidates:
    if (cand / "data" / "processed" / "services").exists():
        PROJECT_ROOT = cand.resolve()
        break

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_SERVICES = DATA_DIR / "processed" / "services"

print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: /Users/rohanraval/Documents/GitHub/Mapping-Youth-Opportunity-Deserts


## Fix 1: ACS Notebook Error

Replace the broken cell in `acs_2023_cleaning_eda.ipynb` with:

In [7]:
print("""CORRECTED CODE:

sex = extract_tract_geoid(sex_raw)

required_sex_cols = [
    "NAME",
    "B01001_001E",
    "B01001_004E", "B01001_005E", "B01001_006E", "B01001_007E",
    "B01001_028E", "B01001_029E", "B01001_030E", "B01001_031E",
]
check_required_columns(sex, required_sex_cols, table_name="B01001")

sex_small = sex[["GEOID", "NAME"] + required_sex_cols[1:]].copy()

for col in sex_small.columns:
    if col not in ["GEOID", "NAME"]:
        sex_small[col] = pd.to_numeric(sex_small[col], errors="coerce")

sex_small["pop_total"] = sex_small["B01001_001E"]
sex_small["youth_5_17"] = (
    sex_small[["B01001_004E", "B01001_005E", "B01001_006E"]].sum(axis=1) +
    sex_small[["B01001_028E", "B01001_029E", "B01001_030E"]].sum(axis=1)
)
sex_small["youth_10_19"] = (
    sex_small[["B01001_005E", "B01001_006E", "B01001_007E"]].sum(axis=1) +
    sex_small[["B01001_029E", "B01001_030E", "B01001_031E"]].sum(axis=1)
)

denom = sex_small["pop_total"].replace({0: np.nan})
sex_small["youth_5_17_per_1k"] = 1000 * sex_small["youth_5_17"] / denom
sex_small["youth_10_19_per_1k"] = 1000 * sex_small["youth_10_19"] / denom

demo_base = sex_small[[
    "GEOID", "NAME", "pop_total", "youth_5_17", "youth_10_19",
    "youth_5_17_per_1k", "youth_10_19_per_1k"
]].copy()

summarize(demo_base, "Demographics base (from B01001)")
""")

CORRECTED CODE:

sex = extract_tract_geoid(sex_raw)

required_sex_cols = [
    "NAME",
    "B01001_001E",
    "B01001_004E", "B01001_005E", "B01001_006E", "B01001_007E",
    "B01001_028E", "B01001_029E", "B01001_030E", "B01001_031E",
]
check_required_columns(sex, required_sex_cols, table_name="B01001")

sex_small = sex[["GEOID", "NAME"] + required_sex_cols[1:]].copy()

for col in sex_small.columns:
    if col not in ["GEOID", "NAME"]:
        sex_small[col] = pd.to_numeric(sex_small[col], errors="coerce")

sex_small["pop_total"] = sex_small["B01001_001E"]
sex_small["youth_5_17"] = (
    sex_small[["B01001_004E", "B01001_005E", "B01001_006E"]].sum(axis=1) +
    sex_small[["B01001_028E", "B01001_029E", "B01001_030E"]].sum(axis=1)
)
sex_small["youth_10_19"] = (
    sex_small[["B01001_005E", "B01001_006E", "B01001_007E"]].sum(axis=1) +
    sex_small[["B01001_029E", "B01001_030E", "B01001_031E"]].sum(axis=1)
)

denom = sex_small["pop_total"].replace({0: np.nan})
sex_small["youth_5_17_per_1k

## Fix 2: Geocode Missing Coordinates

In [8]:
services = pd.read_csv(PROCESSED_SERVICES / "services_master_cleaned.csv", dtype={"zip": str})

print("Total services:", len(services))
print("\nMissing coordinates:")
print(services.groupby("service_type")[["lat", "lon"]].apply(lambda x: x.isna().sum()))

Total services: 520

Missing coordinates:
              lat  lon
service_type          
library       130  130
park           17   17
rec_center      1    1
ymca           23   23


In [9]:
def geocode_address(row, geocoder):
    parts = []
    for field in ["address", "city", "state", "zip"]:
        if pd.notna(row[field]) and str(row[field]) != "nan":
            parts.append(str(row[field]))
    
    if not parts:
        return None, None
    
    try:
        location = geocoder(", ".join(parts))
        if location:
            return location.latitude, location.longitude
    except Exception as e:
        print(f"Error: {e}")
    
    return None, None

def geocode_services(df, service_types=None, batch_size=50):
    geolocator = Nominatim(user_agent="youth_opportunity_deserts_sd")
    geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)
    
    df = df.copy()
    missing = df["lat"].isna() | df["lon"].isna()
    
    if service_types:
        missing = missing & df["service_type"].isin(service_types)
    
    to_geocode = df[missing].copy()
    print(f"Geocoding {len(to_geocode)} services...")
    
    success = fail = 0
    
    for idx, row in to_geocode.iterrows():
        lat, lon = geocode_address(row, geocode)
        
        if lat and lon:
            df.loc[idx, "lat"] = lat
            df.loc[idx, "lon"] = lon
            success += 1
            print(f"✓ {row['name']}: ({lat:.6f}, {lon:.6f})")
        else:
            fail += 1
            print(f"✗ {row['name']}")
        
        if (success + fail) % batch_size == 0:
            print(f"Checkpoint: {success} success, {fail} failed")
            time.sleep(2)
    
    print(f"\nComplete: {success} success, {fail} failed ({100*success/(success+fail):.1f}%)")
    return df

In [10]:
services_geocoded = geocode_services(services, service_types=["library"], batch_size=25)

Geocoding 130 services...
✓ Carlsbad City Library: (33.105698, -117.268960)
✓ Georgina Cole Library: (33.164426, -117.340496)


RateLimiter caught an error, retrying (0/2 tries). Called with (*('3368 Eureka Pl, Carlsbad, CA, 92008',), **{}).
Traceback (most recent call last):
  File "/Users/rohanraval/miniforge3/lib/python3.12/site-packages/urllib3/connectionpool.py", line 534, in _make_request
    response = conn.getresponse()
               ^^^^^^^^^^^^^^^^^^
  File "/Users/rohanraval/miniforge3/lib/python3.12/site-packages/urllib3/connection.py", line 565, in getresponse
    httplib_response = super().getresponse()
                       ^^^^^^^^^^^^^^^^^^^^^
  File "/Users/rohanraval/miniforge3/lib/python3.12/http/client.py", line 1428, in getresponse
    response.begin()
  File "/Users/rohanraval/miniforge3/lib/python3.12/http/client.py", line 331, in begin
    version, status, reason = self._read_status()
                              ^^^^^^^^^^^^^^^^^^^
  File "/Users/rohanraval/miniforge3/lib/python3.12/http/client.py", line 292, in _read_status
    line = str(self.fp.readline(_MAXLINE + 1), "iso-8859-1

✓ Library Learning Center: (33.159296, -117.338561)
✓ Civic Center Branch: (32.640741, -117.082129)
✗ Otay Ranch Branch
✓ South Chula Vista Branch: (32.601734, -117.068450)
✓ Coronado Public: (32.691155, -117.177657)
✓ Escondido Public: (33.120889, -117.079764)
✓ National City Public: (32.670556, -117.104193)
✓ Country Club Lane Senior Center Mini-Library: (33.197305, -117.365307)
✓ El Corazon Senior Center Mini-Library: (33.205966, -117.314822)
✓ Joe Balderrama Recreation Center: (33.204485, -117.371144)
✓ John Landes Community Center Library: (33.195284, -117.285055)
✓ Melba Bishop Recreation Center Mini-Library: (33.255240, -117.287567)
✓ Mission Branch: (33.199646, -117.372964)
✓ Oceanside Civic Center Library: (33.200931, -117.383755)


RateLimiter caught an error, retrying (0/2 tries). Called with (*('804 Pier View Way #101, Oceanside, CA, 92054',), **{}).
Traceback (most recent call last):
  File "/Users/rohanraval/miniforge3/lib/python3.12/site-packages/urllib3/connectionpool.py", line 534, in _make_request
    response = conn.getresponse()
               ^^^^^^^^^^^^^^^^^^
  File "/Users/rohanraval/miniforge3/lib/python3.12/site-packages/urllib3/connection.py", line 565, in getresponse
    httplib_response = super().getresponse()
                       ^^^^^^^^^^^^^^^^^^^^^
  File "/Users/rohanraval/miniforge3/lib/python3.12/http/client.py", line 1428, in getresponse
    response.begin()
  File "/Users/rohanraval/miniforge3/lib/python3.12/http/client.py", line 331, in begin
    version, status, reason = self._read_status()
                              ^^^^^^^^^^^^^^^^^^^
  File "/Users/rohanraval/miniforge3/lib/python3.12/http/client.py", line 292, in _read_status
    line = str(self.fp.readline(_MAXLINE + 1), "i

✗ Oceanside READS Literacy Center
✓ 24/7 Library Kiosk Encinitas: (33.058831, -117.262304)
✓ 24/7 Library To Go - Bonsall: (33.270541, -117.233042)
✗ 24/7 Library To Go - Boulevard
✗ 24/7 Library To Go - County Operations Center
✓ 24/7 Library To Go - South Region: (32.606223, -117.085390)
✓ 4S Ranch Branch: (33.021107, -117.114342)
✓ Alpine Branch: (32.838043, -116.776164)
✓ Bonita-Sunnyside Branch: (32.661429, -117.034271)
Checkpoint: 21 success, 4 failed
✓ Borrego Springs Branch: (33.253938, -116.379675)
✓ Campo-Morena Village Branch: (32.607343, -116.478729)
✓ Cardiff-by-the-Sea Branch: (33.021958, -117.281419)
✗ Casa de Oro Branch
✗ Crest Branch
✓ Del Mar Branch: (32.957802, -117.264357)
✓ Descanso Branch: (32.859707, -116.613827)


...

✓ El Cajon Branch: (32.793062, -116.960744)
✗ Encinitas Branch
✓ Fallbrook Branch: (33.381701, -117.253242)
✓ Fletcher Hills Branch: (32.801753, -116.997569)
✓ Imperial Beach Branch: (32.576894, -117.116249)
✓ Jacumba Branch: (32.617314, -116.187112)
✓ Julian Branch: (33.076733, -116.599675)
✓ La Mesa Branch: (32.766425, -117.023298)
✓ Lakeside Branch: (32.857425, -116.924399)
✓ Lemon Grove Branch: (32.738654, -117.029274)
✓ Lincoln Acres Branch: (32.666536, -117.071335)
✓ Pine Valley Branch: (32.824091, -116.530678)
✓ Potrero Branch: (32.610792, -116.612492)
✓ Poway Branch: (32.955993, -117.045927)
✓ Ramona Branch: (33.039919, -116.873213)
✓ Rancho San Diego Branch: (32.749396, -116.929083)
✓ Rancho Santa Fe Branch: (33.021018, -117.205538)
✓ San Marcos Branch: (33.141264, -117.160381)
Checkpoint: 43 success, 7 failed
✓ Santee Branch: (32.844682, -116.996856)
✓ Solana Beach Branch Earl Warren Middle School: (32.984136, -117.259397)
✓ Spring Valley Branch: (32.712325, -117.002342)
✓ Va

RateLimiter caught an error, retrying (0/2 tries). Called with (*('330 Park Blvd, San Diego, CA, 92101',), **{}).
Traceback (most recent call last):
  File "/Users/rohanraval/miniforge3/lib/python3.12/site-packages/urllib3/connectionpool.py", line 534, in _make_request
    response = conn.getresponse()
               ^^^^^^^^^^^^^^^^^^
  File "/Users/rohanraval/miniforge3/lib/python3.12/site-packages/urllib3/connection.py", line 565, in getresponse
    httplib_response = super().getresponse()
                       ^^^^^^^^^^^^^^^^^^^^^
  File "/Users/rohanraval/miniforge3/lib/python3.12/http/client.py", line 1428, in getresponse
    response.begin()
  File "/Users/rohanraval/miniforge3/lib/python3.12/http/client.py", line 331, in begin
    version, status, reason = self._read_status()
                              ^^^^^^^^^^^^^^^^^^^
  File "/Users/rohanraval/miniforge3/lib/python3.12/http/client.py", line 292, in _read_status
    line = str(self.fp.readline(_MAXLINE + 1), "iso-8859-1

✓ Central Library: (32.708906, -117.154141)
✓ City Heights Performance Annex: (32.747479, -117.100308)
✓ City Heights/Weingart: (32.747479, -117.100308)
✓ City Heights/Weingart Library: (32.747479, -117.100308)
✓ Clairemont: (32.793596, -117.193959)
✓ Clairemont Library: (32.793596, -117.193959)
✓ College-Rolando: (32.769436, -117.056133)
✓ College-Rolando Library: (32.769436, -117.056133)
✓ Kensington - Normal Heights: (32.763027, -117.106865)
✓ Kensington-Normal Heights Library: (32.763027, -117.106865)
✓ La Jolla/Riford: (32.840760, -117.276071)
✓ La Jolla/Riford Library: (32.840760, -117.276071)
Checkpoint: 68 success, 7 failed
✓ Linda Vista: (32.783262, -117.170040)
✓ Linda Vista Library: (32.783262, -117.170040)
✓ Logan Heights: (32.697338, -117.133632)
✓ Logan Heights Library: (32.697338, -117.133632)
✓ Mira Mesa: (32.915169, -117.142718)
✓ Mira Mesa Library: (32.915169, -117.142718)
✓ Mission Hills-Hillcrest/Knox: (32.749671, -117.165356)
✓ Mission Hills-Hillcrest/Knox Library:

RateLimiter caught an error, retrying (0/2 tries). Called with (*('12911 Pacific Pl, San Diego, CA, 92130',), **{}).
Traceback (most recent call last):
  File "/Users/rohanraval/miniforge3/lib/python3.12/site-packages/urllib3/connectionpool.py", line 534, in _make_request
    response = conn.getresponse()
               ^^^^^^^^^^^^^^^^^^
  File "/Users/rohanraval/miniforge3/lib/python3.12/site-packages/urllib3/connection.py", line 565, in getresponse
    httplib_response = super().getresponse()
                       ^^^^^^^^^^^^^^^^^^^^^
  File "/Users/rohanraval/miniforge3/lib/python3.12/http/client.py", line 1428, in getresponse
    response.begin()
  File "/Users/rohanraval/miniforge3/lib/python3.12/http/client.py", line 331, in begin
    version, status, reason = self._read_status()
                              ^^^^^^^^^^^^^^^^^^^
  File "/Users/rohanraval/miniforge3/lib/python3.12/http/client.py", line 292, in _read_status
    line = str(self.fp.readline(_MAXLINE + 1), "iso-885

✓ Pacific Highlands Ranch Library: (32.961348, -117.187504)
✓ Paradise Hills: (32.672709, -117.061217)
✓ Paradise Hills Library: (32.672709, -117.061217)
✓ Point Loma/Hervey: (32.740080, -117.229418)
✓ Point Loma/Hervey Library: (32.740080, -117.229418)
✓ Rancho Bernardo: (33.024948, -117.075278)
✓ Rancho Bernardo Library: (33.024948, -117.075278)
✓ Rancho Penasquitos: (32.958029, -117.122208)
✓ Rancho Peñasquitos Library: (32.958029, -117.122208)
✓ San Carlos: (32.802036, -117.040599)
✓ San Carlos Library: (32.802036, -117.040599)
✓ San Diego Central: (32.708906, -117.154141)
✓ San Ysidro: (32.557831, -117.043272)
✓ San Ysidro Library: (32.557831, -117.043272)
✓ Scripps Miramar Ranch: (32.911774, -117.101723)
✓ Scripps Miramar Ranch Library: (32.911774, -117.101723)
✓ Serra Mesa-Kearny Mesa: (32.809207, -117.133705)
✓ Serra Mesa-Kearny Mesa Library: (32.809207, -117.133705)
✓ Skyline Hills: (32.688668, -117.047535)
✓ Skyline Hills Library: (32.688668, -117.047535)
✓ Tierrasanta: (32.8

In [11]:
services_geocoded = geocode_services(services_geocoded, service_types=["ymca"], batch_size=25)

Geocoding 23 services...
✓ Border View Family YMCA: (32.578214, -117.057056)
✓ Cameron Family YMCA: (32.850637, -116.977637)
✓ Copley-Price Family YMCA: (32.755532, -117.101353)
✓ Dan McKinney Family YMCA: (32.858359, -117.242704)
✓ Escondido YMCA – A Campus for Community Well-Being: (33.134763, -117.087185)
✓ Jackie Robinson Family YMCA: (32.705183, -117.097497)
✓ Joe and Mary Mottino Family YMCA: (33.224318, -117.290825)
✓ John A. Davis Family YMCA: (32.790806, -117.006827)
✓ Magdalena Ecke Family YMCA: (33.051395, -117.286596)
✓ McGrath Family YMCA: (32.736939, -116.941369)
✓ Mission Valley YMCA: (32.763497, -117.192389)
✓ Rancho Family YMCA: (32.958056, -117.123814)
✓ Shepherd YMCA Firehouse: (32.846541, -117.273037)
✓ South Bay Family YMCA: (32.639352, -117.011812)
✓ T. Claude and Gladys B. Ryan Family YMCA: (32.749732, -117.232811)
✓ Toby Wells YMCA: (32.829830, -117.130259)
✓ YMCA Camp Marston: (33.042944, -116.621574)
✓ YMCA Camp Surf: (32.585377, -117.128281)
✓ YMCA Childcare 

RateLimiter caught an error, retrying (0/2 tries). Called with (*('4451 30th St Suite 200, San Diego, CA, 92116',), **{}).
Traceback (most recent call last):
  File "/Users/rohanraval/miniforge3/lib/python3.12/site-packages/urllib3/connectionpool.py", line 534, in _make_request
    response = conn.getresponse()
               ^^^^^^^^^^^^^^^^^^
  File "/Users/rohanraval/miniforge3/lib/python3.12/site-packages/urllib3/connection.py", line 565, in getresponse
    httplib_response = super().getresponse()
                       ^^^^^^^^^^^^^^^^^^^^^
  File "/Users/rohanraval/miniforge3/lib/python3.12/http/client.py", line 1428, in getresponse
    response.begin()
  File "/Users/rohanraval/miniforge3/lib/python3.12/http/client.py", line 331, in begin
    version, status, reason = self._read_status()
                              ^^^^^^^^^^^^^^^^^^^
  File "/Users/rohanraval/miniforge3/lib/python3.12/http/client.py", line 292, in _read_status
    line = str(self.fp.readline(_MAXLINE + 1), "i

✗ YMCA Expanded Learning Programs
✓ YMCA of San Diego County: Corporate Office: (32.810608, -117.119642)
✓ YMCA Raintree Ranch: (33.048225, -116.615506)
✓ YMCA Youth and Family Services: (32.757201, -117.131168)

Complete: 22 success, 1 failed (95.7%)


In [12]:
print("Missing after geocoding:")
print(services_geocoded.groupby("service_type")[["lat", "lon"]].apply(lambda x: x.isna().sum()))

valid = services_geocoded.dropna(subset=["lat", "lon"])
in_bounds = (
    (valid["lat"] >= 32.0) & (valid["lat"] <= 33.5) &
    (valid["lon"] >= -117.7) & (valid["lon"] <= -116.0)
)
print(f"\nIn SD bounds: {in_bounds.sum()} / {len(valid)}")

if not in_bounds.all():
    print("\nOutliers:")
    print(valid[~in_bounds][["name", "city", "lat", "lon"]])

Missing after geocoding:
              lat  lon
service_type          
library         7    7
park           17   17
rec_center      1    1
ymca            1    1

In SD bounds: 494 / 494


In [13]:
output = PROCESSED_SERVICES / "services_master_geocoded.csv"
services_geocoded.to_csv(output, index=False)
print(f"Saved: {output}")

Saved: /Users/rohanraval/Documents/GitHub/Mapping-Youth-Opportunity-Deserts/data/processed/services/services_master_geocoded.csv


## Fix 3: Enhanced Youth-Serving Identification

In [14]:
def identify_youth_serving(df):
    df = df.copy()
    
    keywords = [
        r'\byouth\b', r'\bteen\b', r'\bjunior\b', r'\badolescent\b',
        r'\bk-12\b', r'\bk12\b', r'\b5-17\b', r'\b10-19\b',
        r'\bafter.?school\b', r'\bsummer.?program\b', r'\bcamp\b',
        r'\btutoring\b', r'\bhomework\b',
        r'\bboys.?&.?girls\b', r'\bb&g\b', r'\bbgc\b',
        r'\bymca\b', r'\bywca\b', r'\bscouts\b', r'\b4-h\b',
        r'\bplayground\b', r'\btot.?lot\b', r'\bskate.?park\b',
        r'\bmentor\b', r'\bbasketball\b'
    ]
    
    name_lower = df["name"].astype(str).str.lower()
    org_lower = df["org"].astype(str).str.lower()
    
    df["youth_serving"] = False
    
    for kw in keywords:
        matches = (
            name_lower.str.contains(kw, case=False, na=False, regex=True) |
            org_lower.str.contains(kw, case=False, na=False, regex=True)
        )
        df.loc[matches, "youth_serving"] = True
    
    df.loc[df["service_type"].isin(["library", "rec_center"]), "youth_serving"] = True
    
    if "playground" in df.columns:
        df.loc[(df["service_type"] == "park") & (df["playground"] > 0), "youth_serving"] = True
    if "tot_lot" in df.columns:
        df.loc[(df["service_type"] == "park") & (df["tot_lot"] > 0), "youth_serving"] = True
    
    return df

In [15]:
services_enhanced = identify_youth_serving(services_geocoded)

print("Youth-serving by type:")
print(services_enhanced.groupby("service_type")["youth_serving"].sum())
print(f"\nTotal: {services_enhanced['youth_serving'].sum()} / {len(services_enhanced)}")
print(f"Percentage: {100*services_enhanced['youth_serving'].sum()/len(services_enhanced):.1f}%")

Youth-serving by type:
service_type
library       130
park            2
rec_center     61
ymca           23
Name: youth_serving, dtype: int64

Total: 216 / 520
Percentage: 41.5%


In [16]:
output = PROCESSED_SERVICES / "services_master_enhanced.csv"
services_enhanced.to_csv(output, index=False)
print(f"Saved: {output}")

Saved: /Users/rohanraval/Documents/GitHub/Mapping-Youth-Opportunity-Deserts/data/processed/services/services_master_enhanced.csv
